# Memoization and Tabulation

```{contents}
:local:
:depth: 2
```

A state identifies one smaller problem. A transition relates it to other states. Define the state, base cases, legal inputs, dependency order, and returned result before choosing a cache or table. Counting problems such as Fibonacci need a decomposition; minimum-coin optimization additionally needs optimal substructure.

## Repeated Work Is Visible

```{index} Repeated Work Is Visible
```

F(0)=0, F(1)=1, and F(n)=F(n−1)+F(n−2) for n≥2. The argument strictly decreases, but the recursion tree repeats arguments. Count every invocation, including base cases; computed counts only non-base expansions. Compare the same n, not unrelated examples.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var naive=FibPlanning.Fibonacci(8,false);
var memo=FibPlanning.Fibonacci(8);
Console.WriteLine($"naive: value={naive.Value}, calls={naive.Calls}, computed={naive.Computed}");
Console.WriteLine($"memo: value={memo.Value}, calls={memo.Calls}, hits={memo.Hits}, computed={memo.Computed}");
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


The naive call recurrence is C(0)=C(1)=1 and C(n)=1+C(n−1)+C(n−2); C(n)=2F(n+1)−1. Calls are exponential in n while active stack depth is Θ(n). This teaching implementation caps naive n at 25 to keep browser runs short; all Fibonacci APIs otherwise accept 0..92, the range that fits signed long. Invalid n is rejected before recursion.

## Memoization Stores Complete States

```{index} Memoization Stores Complete States
```

The cache key n fully identifies F(n) because the recurrence and base cases are fixed. Every cached value is a completed correct result. Compute a missing state recursively, then store it. Local cache ownership avoids mixing unrelated requests or caller-supplied incorrect entries.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int n in new[]{0,1,2,8,40})
{
 var result=FibPlanning.Fibonacci(n);
 Console.WriteLine($"n={n}, value={result.Value}, calls={result.Calls}, hits={result.Hits}, computed={result.Computed}");
}
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


For n≥2 there are n−1 distinct non-base states, 2n−1 calls including cache hits, and max(0,n−3) hits. Base results are not cached here. Expected constant-time dictionary lookup gives Θ(n) time, Θ(n) cache storage, and Θ(n) stack space. These are distinct costs. A fresh invocation starts a fresh cache; memoization does not make the recursive stack disappear. Induction proves correctness: bases are correct, recursive dependencies are smaller, and their sum is F(n).

## Tabulation Follows Dependency Order

```{index} Tabulation Follows Dependency Order
```

The invariant before iteration i is that entries 0..i−1 equal their specified Fibonacci values. Writing table[i] from i−1 and i−2 preserves it. Initialize entry one only when it exists; zero must work.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

foreach(int n in new[]{0,1,10})
{
 var table=FibPlanning.FibonacciTable(n);
 Console.WriteLine($"n={n}: [{string.Join(",",table)}], rolling={FibPlanning.FibonacciRolling(n)}");
}
Console.WriteLine($"F(92)={FibPlanning.FibonacciRolling(92)}");
try{FibPlanning.FibonacciTable(93);}catch(ArgumentOutOfRangeException){Console.WriteLine("overflow domain rejected");}
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


A full table costs Θ(n) time and Θ(n) storage without recursion. Rolling state keeps only the two needed predecessors, giving Θ(n) time and Θ(1) auxiliary storage when only the final value is requested. It cannot provide every historical table entry. Checked addition protects arithmetic; validating 0..92 also avoids computing an unnecessary F(93) on the final rolling update.

## Minimum Coins: Model Before Implementation

```{index} Minimum Coins: Model Before Implementation
```

A purchasing system uses unlimited packs of sizes 1,3,4. Minimize the number of packs totaling target six. Let B(a) be the minimum count for a; B(0)=0, and B(a)=min(1+B(a−c)) over positive denominations c≤a whose remainder is reachable. If no such predecessor exists, a is unreachable. Removing the last coin from any optimal plan leaves an optimal remainder plan: a better remainder would improve the original plan.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var plan=CoinPlanning.MinimumCoins(new[]{1,3,4},6);
Console.WriteLine($"minimum={plan.Count}, coins={string.Join("+",plan.Coins)}");
Console.WriteLine("table="+string.Join(",",plan.Best.Select(x=>x?.ToString()??"unreachable")));
var impossible=CoinPlanning.MinimumCoins(new[]{4,6},7);
Console.WriteLine($"seven reachable={impossible.Count is not null}, witness length={impossible.Coins.Length}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


Positive denominations ensure a−c<a, so amounts in increasing order satisfy dependencies and reconstruction strictly decreases remaining amount. Nullable count distinguishes unreachable from a valid zero-coin plan. Duplicate denominations are normalized; ascending denominations and strict improvement make equal-count ties deterministic. Empty denominations can make only zero. Reject zero/negative coins: a zero coin breaks progress and a negative coin changes the state graph. The table invariant is that all smaller amounts have correct optima or correct unreachable markers; considering every possible last coin establishes the next entry.

## State Size, Numeric Size, and the Requested Result

```{index} State Size, Numeric Size, and the Requested Result
```

With target T and D input denominations, sorting/normalization costs O(D log(D+1)); filling the table costs O(Td) for d distinct denominations, and table storage is O(T). Reconstructing K chosen coins adds O(K) output space and time. This implementation caps T at one million as an explicit allocation policy. These costs depend on the numeric target, not just the number of digits encoding it: the method is pseudo-polynomial.

Memoization can avoid some states reachable from the requested state; tabulation can avoid recursion and simplify dependency order. Neither is universally better. An amount-only coin state is sufficient for unlimited coins. If each pack is available once, remaining inventory matters: amount alone loses essential information. Chapter 24’s subset search uses (position, remaining), which preserves which items are still available. Rolling storage also trades away reconstruction information unless another recovery method is supplied.

## Exercise

```{index} Exercise
```

Check zero, unreachable, duplicate, and invalid denomination contracts. Explain why unlimited coins require a different state from one-use invoices.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] denominations={3,3,4};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] denominations={3,3,4};
foreach(int target in new[]{0,2,6})
{
 var p=CoinPlanning.MinimumCoins(denominations,target);
 Console.WriteLine($"target={target}, count={p.Count?.ToString()??"unreachable"}, witness=[{string.Join(",",p.Coins)}]");
}
try{CoinPlanning.MinimumCoins(new[]{0,3},6);}catch(ArgumentException){Console.WriteLine("zero denomination rejected");}
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


target=0, count=0, witness=[]
target=2, count=unreachable, witness=[]
target=6, count=2, witness=[3,3]
zero denomination rejected


```{rubric} Footnotes
```
[^1]: A correct state contains all information relevant to future choices. Reusing results for different states can silently change the problem.
[^2]: Complexity claims here count fixed-width arithmetic as constant time; arbitrary-precision counting requires accounting for growing integer sizes.